# Создание датасета с координатами паттерна "Голова и плечи"

В этом ноутбуке мы создадим полноценный датасет с использованием существующего кода для обнаружения паттернов и будем сохранять не только информацию о наличии паттерна, но и его координаты (начало и конец).

## 1. Загрузка необходимых библиотек

In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import yfinance as yf

## 2. Импорт пользовательских модулей

In [8]:
from data_loader import DataLoader
from pattern_detector import HeadShouldersDetector
from dataset_builder import DatasetBuilder
from dataset_pipeline import build_dataset_for_ticker, save_csv

## 3. Конфигурация параметров датасета

In [9]:
WINDOW_SIZE = 120
STEP = 5

DETECTOR_PARAMS = {
    'order': 5,
    'shoulder_tol': 0.25,
    'min_head_height': 0.02,
    'min_distance': 8,
    'max_width': 120
}

TICKERS = ["AAPL", 'MSFT', 'GOOGL', 'TSLA']
DATE_RANGE = {'start': '2015-01-01', 'end': '2024-01-01'}
TEST_SIZE = 0.2

In [10]:
X_all = []
y_coords_all = []
successful_tickers = []

for ticker in TICKERS:
    print(f"Загрузка данных для {ticker}...")
    try:
        X, y_labels, y_coords = build_dataset_for_ticker(ticker, window_size=WINDOW_SIZE)
        
        if len(X) > 0:
            X_all.append(X)
            y_coords_all.append(y_coords)
            successful_tickers.append(ticker)
            print(f"✓ Успешно: {len(X)} окон, паттернов: {int(y_coords[:, 0].sum())}")
        else:
            print(f"⚠ Нет данных для {ticker}")
    except Exception as e:
        print(f"✗ Ошибка для {ticker}: {str(e)[:80]}")

if X_all and y_coords_all:
    X_combined = np.concatenate(X_all, axis=0)
    y_coords_combined = np.concatenate(y_coords_all, axis=0)
    print(f"\nВсего окон: {len(X_combined)}, с паттернами: {int(y_coords_combined[:, 0].sum())}")
else:
    print("❌ Не удалось загрузить данные")
    X_combined = None
    y_coords_combined = None

Загрузка данных для AAPL...



1 Failed download:
['AAPL']: TypeError("'NoneType' object is not subscriptable")


AAPL: empty dataframe
✗ Ошибка для AAPL: 'close'
Загрузка данных для MSFT...



1 Failed download:
['MSFT']: TypeError("'NoneType' object is not subscriptable")


MSFT: empty dataframe
✗ Ошибка для MSFT: 'close'
Загрузка данных для GOOGL...



1 Failed download:
['GOOGL']: TypeError("'NoneType' object is not subscriptable")


GOOGL: empty dataframe
✗ Ошибка для GOOGL: 'close'
Загрузка данных для TSLA...



1 Failed download:
['TSLA']: TypeError("'NoneType' object is not subscriptable")


TSLA: empty dataframe
✗ Ошибка для TSLA: 'close'
❌ Не удалось загрузить данные


## 5. Разделение на обучающую и тестовую выборки

In [7]:
if X_combined is not None and y_coords_combined is not None:
    X_train, X_test, y_coords_train, y_coords_test = train_test_split(
        X_combined, 
        y_coords_combined, 
        test_size=TEST_SIZE, 
        random_state=RANDOM_STATE,
        stratify=y_coords_combined[:, 0]
    )
    print(f"✓ Обучение: {len(X_train)} строк, Тест: {len(X_test)} строк")
else:
    print("❌ Ошибка: данные не загружены")
    X_train = X_test = y_coords_train = y_coords_test = None

NameError: name 'X_combined' is not defined

## 6. Сохранение датасета в CSV файлы

In [ ]:
if X_train is not None and y_coords_train is not None:
    save_csv(X_train, y_coords_train, 'train.csv')
    save_csv(X_test, y_coords_test, 'test.csv')
    print("✓ Датасет сохранён: train.csv, test.csv")
else:
    print("❌ Ошибка: невозможно сохранить датасет")

## 7. Проверка выходного датасета

In [4]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

print(f"Обучение: {len(train_df)} строк")
print(f"Тест: {len(test_df)} строк")
print(f"Всего: {len(train_df) + len(test_df)} строк")
print(f"\n✓ Датасет успешно создан")

Обучение: 1372 строк
Тест: 344 строк
Всего: 1716 строк

✓ Датасет успешно создан


## 8. Визуализация примеров из датасета

In [6]:
import matplotlib.pyplot as plt

# Отделяем ряды с паттернами и без
indices_with_pattern = np.where(y_coords_test[:, 0] == 1)[0]
indices_without_pattern = np.where(y_coords_test[:, 0] == 0)[0]

# Выбираем случайные примеры
num_with = min(3, len(indices_with_pattern))
num_without = min(3, len(indices_without_pattern))

sample_indices_with = np.random.choice(indices_with_pattern, size=num_with, replace=False)
sample_indices_without = np.random.choice(indices_without_pattern, size=num_without, replace=False)

# Создаём визуализацию
fig, axes = plt.subplots(num_with + num_without, 1, figsize=(14, 4 * (num_with + num_without)))
if num_with + num_without == 1:
    axes = [axes]

plot_idx = 0

# Визуализируем примеры С паттернами
for idx in sample_indices_with:
    ax = axes[plot_idx]
    time_series = X_test[idx]
    has_pattern, start, end = y_coords_test[idx]  # Правильный порядок!
    
    # Денормализуем координаты (они хранятся в диапазоне [0, 1])
    start = int(start * len(time_series))
    end = int(end * len(time_series))
    
    ax.plot(time_series, 'b-', linewidth=1.5, label='Цена')
    ax.axvline(x=start, color='g', linestyle='--', linewidth=2, label=f'Начало паттерна ({start})')
    ax.axvline(x=end, color='r', linestyle='--', linewidth=2, label=f'Конец паттерна ({end})')
    
    # Заполнение только если координаты корректны
    if 0 <= start < len(time_series) and 0 <= end < len(time_series) and start <= end:
        ax.fill_between(range(start, end + 1), time_series[start:end + 1].min() - 1, 
                         time_series[start:end + 1].max() + 1, alpha=0.2, color='yellow')
    
    ax.set_title(f'Пример ряда С паттерном "Голова и плечи"', fontsize=12, fontweight='bold')
    ax.set_xlabel('Время (бары)')
    ax.set_ylabel('Нормализованная цена')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    plot_idx += 1

# Визуализируем примеры БЕЗ паттернов
for idx in sample_indices_without:
    ax = axes[plot_idx]
    time_series = X_test[idx]
    
    ax.plot(time_series, 'b-', linewidth=1.5, label='Цена')
    ax.set_title(f'Пример ряда БЕЗ паттерна', fontsize=12, fontweight='bold')
    ax.set_xlabel('Время (бары)')
    ax.set_ylabel('Нормализованная цена')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    plot_idx += 1

plt.tight_layout()
plt.show()

print(f"\n✓ Визуализация завершена")
print(f"  - Рядов с паттернами: {num_with}")
print(f"  - Рядов без паттернов: {num_without}")

NameError: name 'y_coords_test' is not defined